# Tuning ligero para Random Forest y RBF-SVC

Este notebook reduce el coste computacional del tuning:

- usa `RandomizedSearchCV` por defecto
- evita `n_jobs=-1`
- permite tunear un modelo cada vez
- usa `StratifiedGroupKFold` para respetar sujetos

Está pensado para integrarse con tu proyecto actual.


In [17]:

import pandas as pd
from sklearn.base import clone
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedGroupKFold
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DEMO_PATH = PROJECT_ROOT / "demo"

if str(DEMO_PATH) not in sys.path:
    sys.path.append(str(DEMO_PATH))


## Funciones auxiliares

In [18]:

def add_pipeline_prefix(estimator, base_grid):
    if hasattr(estimator, "named_steps"):
        last_step_name = list(estimator.named_steps.keys())[-1]
        return {f"{last_step_name}__{k}": v for k, v in base_grid.items()}
    return base_grid


def tune_models(
    models,
    param_grids,
    X,
    y,
    groups,
    model_names=("random_forest", "rbf_svc"),
    cv_splits=5,
    scoring="balanced_accuracy",
    search_type="random",   # "random" o "grid"
    n_iter=12,              # solo para random search
    n_jobs=1,
    verbose=2,
):
    cv = StratifiedGroupKFold(
        n_splits=cv_splits,
        shuffle=True,
        random_state=42,
    )

    results = []
    best_estimators = {}

    for model_name in model_names:
        print(f"\n=== TUNING: {model_name} ===")

        model = clone(models[model_name])
        grid = add_pipeline_prefix(model, param_grids[model_name])

        if search_type == "grid":
            search = GridSearchCV(
                estimator=model,
                param_grid=grid,
                scoring=scoring,
                cv=cv,
                n_jobs=n_jobs,
                refit=True,
                verbose=verbose,
                error_score="raise",
            )
        else:
            search = RandomizedSearchCV(
                estimator=model,
                param_distributions=grid,
                n_iter=n_iter,
                scoring=scoring,
                cv=cv,
                n_jobs=n_jobs,
                refit=True,
                random_state=42,
                verbose=verbose,
                error_score="raise",
            )

        search.fit(X, y, groups=groups)

        best_estimators[model_name] = search.best_estimator_

        results.append({
            "Modelo": model_name,
            "BestScore": search.best_score_,
            "BestParams": search.best_params_,
        })

        print("Best score:", round(search.best_score_, 4))
        print("Best params:", search.best_params_)

    results_df = pd.DataFrame(results).sort_values(by="BestScore", ascending=False)
    return results_df, best_estimators


## Importar tu pipeline y definir grids pequeños

Estos grids están recortados para que el PC no sufra tanto.

In [19]:

from pipeline import get_models

models = get_models()

param_grids = {
    "random_forest": {
        "n_estimators": [200, 500],
        "max_depth": [None, 20],
        "min_samples_leaf": [1, 2],
        "max_features": ["sqrt", 0.5],
        "class_weight": ["balanced"],
    },
    "rbf_svc": {
        "C": [1, 10],
        "gamma": ["scale", 0.01],
        "class_weight": ["balanced"],
    },
}


In [20]:
from pathlib import Path
import pandas as pd

from data_load import load_dataset
from preprocessing import preprocess_dataset
from epochs import create_epochs
from features import extract_epoch_features
from spectral_features import extract_spectral_features

CSV_PATH = Path.cwd().parent / "data" / "adhdata.csv"

df = load_dataset(CSV_PATH)
df_clean, eeg_cols = preprocess_dataset(df)

X_epochs, y_epochs, groups_epochs = create_epochs(
    df=df_clean,
    eeg_columns=eeg_cols,
    label_column="Class",
    group_column="ID",
    epoch_size=1280,
    step_size=640,
)

X_time = extract_epoch_features(X_epochs, eeg_cols)

X_spectral = extract_spectral_features(
    X_epochs=X_epochs,
    channel_names=eeg_cols,
    sfreq=128,
    nperseg=1280,
)

X_features = pd.concat(
    [X_time.reset_index(drop=True), X_spectral.reset_index(drop=True)],
    axis=1,
)

print("X_epochs:", X_epochs.shape)
print("X_time:", X_time.shape)
print("X_spectral:", X_spectral.shape)
print("X_features:", X_features.shape)
print("y_epochs:", y_epochs.shape)
print("groups_epochs:", groups_epochs.shape)

X_epochs: (3201, 1280, 19)
X_time: (3201, 228)
X_spectral: (3201, 230)
X_features: (3201, 458)
y_epochs: (3201,)
groups_epochs: (3201,)


## Ejecutar tuning

Aquí debes tener ya creados:

- `X_features`
- `y_epochs`
- `groups_epochs`

Puedes tunear ambos modelos o solo uno.

In [21]:

tuning_results_df, best_estimators = tune_models(
    models=models,
    param_grids=param_grids,
    X=X_features,
    y=y_epochs,
    groups=groups_epochs,
    model_names=("random_forest", "rbf_svc"),
    cv_splits=5,
    scoring="balanced_accuracy",
    search_type="grid",   
    n_iter=10,              
    n_jobs=1,               
    verbose=2,
)

tuning_results_df



=== TUNING: random_forest ===
Fitting 5 folds for each of 16 candidates, totalling 80 fits
[CV] END model__class_weight=balanced, model__max_depth=None, model__max_features=sqrt, model__min_samples_leaf=1, model__n_estimators=200; total time=   3.8s
[CV] END model__class_weight=balanced, model__max_depth=None, model__max_features=sqrt, model__min_samples_leaf=1, model__n_estimators=200; total time=   3.5s
[CV] END model__class_weight=balanced, model__max_depth=None, model__max_features=sqrt, model__min_samples_leaf=1, model__n_estimators=200; total time=   4.1s
[CV] END model__class_weight=balanced, model__max_depth=None, model__max_features=sqrt, model__min_samples_leaf=1, model__n_estimators=200; total time=   3.9s
[CV] END model__class_weight=balanced, model__max_depth=None, model__max_features=sqrt, model__min_samples_leaf=1, model__n_estimators=200; total time=   4.1s
[CV] END model__class_weight=balanced, model__max_depth=None, model__max_features=sqrt, model__min_samples_leaf=1

,Modelo,BestScore,BestParams
0,random_forest,0.779311,"{'model__class_weight': 'balanced', 'model__ma..."
1,rbf_svc,0.739314,"{'model__C': 10, 'model__class_weight': 'balan..."


## Qué mirar

- `BestScore`: mejor balanced accuracy media en CV
- `BestParams`: hiperparámetros ganadores
- `best_estimators`: modelos ya ajustados


